In [ ]:
!pip install -q -U trl torchao bert-score git+https://github.com/bckim92/language-evaluation.git

# Uncomment if running in Google Colab:
# from google.colab import drive
# drive.mount("/content/drive")

import os, gc, re, json, math, random, zipfile, tarfile, urllib.request, subprocess
from pathlib import Path

import gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, Image as HFImage
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer
from trl.trainer.dpo_trainer import DataCollatorForVisionPreference

from bert_score import score as bertscore_score
from language_evaluation.coco_caption_py3.pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from language_evaluation.coco_caption_py3.pycocoevalcap.bleu.bleu import Bleu
from language_evaluation.coco_caption_py3.pycocoevalcap.rouge.rouge import Rouge
from language_evaluation.coco_caption_py3.pycocoevalcap.cider.cider import Cider
import language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor as meteor_mod

# ============================================================
# 1. EXPERIMENT CONFIGURATION
# ============================================================
SEED = 10623
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
PROJECT_DIR = Path("/content/drive/MyDrive/EMMbedding")
DATA_DIR = Path("/content/data/artcap")

EXP_NAME = "E3_ranked_dpo"
PAIRS_PATH = PROJECT_DIR / "judge_pairs.jsonl"
EXP_ROOT = PROJECT_DIR / EXP_NAME

MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 8

# Winning configuration (lr=5e-6, beta=0.05, ep=1) + sweep candidates
SWEEP_CONFIGS = [
    {"learning_rate": 5e-6, "beta": 0.1,  "num_epochs": 1},  # Baseline DPO
    {"learning_rate": 1e-5, "beta": 0.1,  "num_epochs": 1},  # Stronger policy shift
    {"learning_rate": 5e-6, "beta": 0.05, "num_epochs": 1},  # Weaker KL anchor (Winner)
    {"learning_rate": 5e-6, "beta": 0.1,  "num_epochs": 2},  # 2-epoch schedule
]

for d in [DATA_DIR, EXP_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
PROMPT = """Describe the visual content of this painting in one concise, informative sentence.

Focus on:
- The people, objects, and subjects depicted.
- Their actions and spatial relationships.
- Important colors, visual attributes, and setting.

Only describe details that are visually supported by the image.
Do not speculate about the artist, historical context, or symbolic meaning.

Return only the caption."""

def ensure_images_extracted():
    img_dir = DATA_DIR / "image_archive"
    if not img_dir.exists() or not any(img_dir.iterdir()):
        archive = DATA_DIR / "artcap_images_archive"
        if not archive.exists():
            gdown.download(
                "https://drive.google.com/uc?id=1inQ4W-CtNhhfo1vAehSbLgyMPTmltWVa",
                str(archive), quiet=False, fuzzy=True
            )
        img_dir.mkdir(parents=True, exist_ok=True)
        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as z: z.extractall(img_dir)
        else:
            with tarfile.open(archive) as t: t.extractall(img_dir)
    return {p.stem: str(p) for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}

image_lookup = ensure_images_extracted()
CAPTIONS_JSONL = PROJECT_DIR / "captions.jsonl"

if CAPTIONS_JSONL.exists():
    all_rows = [json.loads(line) for line in open(CAPTIONS_JSONL) if line.strip()]
    for r in all_rows:
        stem = str(r["image_id"])
        r["image_path"] = image_lookup.get(stem) or image_lookup[str(int(stem))]
else:
    kale_archive = DATA_DIR / "kale_data.zip"
    kale_dir = DATA_DIR / "kale"
    if not kale_dir.exists():
        gdown.download(
            "https://drive.google.com/uc?id=1OdPZMQQACgy2OmRsIp3-W-KnD7ATC-hO",
            str(kale_archive), quiet=False, fuzzy=True
        )
        with zipfile.ZipFile(kale_archive) as z: z.extractall(kale_dir)

    def load_split_dict(filename):
        raw = json.load(open(next(kale_dir.rglob(filename))))
        return {Path(str(img_id)).stem: [str(c) for c in caps if c] for img_id, caps in raw}

    tv_ann = load_split_dict("artcap_train_val.json")
    test_ann = load_split_dict("artcap_test.json")
    train_ids, val_ids = train_test_split(list(tv_ann.keys()), test_size=0.1, random_state=SEED)
    train_ids, val_ids = set(train_ids), set(val_ids)

    def build_rows(ann_subset, split):
        rows = []
        for img_id, refs in ann_subset.items():
            path = image_lookup.get(img_id) or image_lookup[str(int(img_id))]
            rows.append({"image_id": str(img_id), "image_path": str(path), "split": split, "references": refs})
        return rows

    all_rows = (
        build_rows({k: tv_ann[k] for k in train_ids}, "train")
        + build_rows({k: tv_ann[k] for k in val_ids}, "val")
        + build_rows(test_ann, "test")
    )
    with open(CAPTIONS_JSONL, "w") as f:
        for r in all_rows: f.write(json.dumps(r) + "\n")

train_rows = [r for r in all_rows if r["split"] == "train"]
val_rows   = [r for r in all_rows if r["split"] == "val"]
test_rows  = [r for r in all_rows if r["split"] == "test"]

path_by_id = {str(r["image_id"]): r["image_path"] for r in all_rows}
raw_pairs = [json.loads(line) for line in open(PAIRS_PATH) if line.strip()]

clean_pairs = [
    {**r, "image_path": path_by_id[str(r["image_id"])]}
    for r in raw_pairs
    if r["chosen"].strip() != r["rejected"].strip()
]

def to_hf_dataset(pairs):
    return Dataset.from_list([
        {
            "image": {"path": r["image_path"], "bytes": None},
            "prompt": [{"role": "user", "content": PROMPT}],
            "chosen": [{"role": "assistant", "content": r["chosen"]}],
            "rejected": [{"role": "assistant", "content": r["rejected"]}],
            "image_id": str(r["image_id"]),
        }
        for r in pairs
    ]).cast_column("image", HFImage())

train_dataset = to_hf_dataset([r for r in clean_pairs if r["split"] == "train"])
val_dataset   = to_hf_dataset([r for r in clean_pairs if r["split"] == "val"])
print(f"Paintings -> Train: {len(train_rows)}, Val: {len(val_rows)}, Test: {len(test_rows)}")
print(f"Preference Pairs -> Train: {len(train_dataset)}, Val: {len(val_dataset)}")

In [ ]:
meteor_dir = os.path.dirname(meteor_mod.__file__)
data_dir = os.path.join(meteor_dir, "data")
os.makedirs(data_dir, exist_ok=True)
paraphrase_path = os.path.join(data_dir, "paraphrase-en.gz")
if not os.path.exists(paraphrase_path) or os.path.getsize(paraphrase_path) == 0:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/tylin/coco-caption/master/pycocoevalcap/meteor/data/paraphrase-en.gz",
        paraphrase_path,
    )

def init_model_and_processor(lora_r=16, lora_alpha=32, lora_dropout=0.05):
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        size={"shortest_edge": 224 * 28 * 28, "longest_edge": 512 * 28 * 28},
    )
    processor.tokenizer.padding_side = "left"
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False

    for name, param in model.named_parameters():
        if "visual" in name.lower():
            param.requires_grad = False

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=r".*language_model.*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    )
    return model, processor, lora_config

def compute_meteor_clean(gts, res):
    jar_path = os.path.join(meteor_dir, "meteor-1.5.jar")
    proc = subprocess.Popen(
        ["java", "-Xmx2G", "-XX:-UseContainerSupport", "-Xlog:all=off", "-jar", jar_path, "-", "-", "-stdio", "-l", "en", "-norm"],
        cwd=meteor_dir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
    )
    imgIds = list(gts.keys())
    eval_line = "EVAL"
    for i in imgIds:
        hyp = res[i][0].replace("|||", "").replace("  ", " ")
        refs_str = " ||| ".join(gts[i])
        proc.stdin.write(f"SCORE ||| {refs_str} ||| {hyp}\n".encode("utf-8"))
        proc.stdin.flush()
        stat = proc.stdout.readline().decode("utf-8").strip()
        eval_line += f" ||| {stat}"
    proc.stdin.write(f"{eval_line}\n".encode("utf-8"))
    proc.stdin.flush()
    _ = [proc.stdout.readline().decode("utf-8").strip() for _ in imgIds]
    agg_line = proc.stdout.readline().decode("utf-8").strip()
    if " " in agg_line:
        proc.stdin.write(f"SCORE ||| {agg_line}\n".encode("utf-8"))
        proc.stdin.flush()
        score = float(proc.stdout.readline().decode("utf-8").strip())
    else:
        score = float(agg_line)
    proc.kill()
    return score

def eval_adapter_split(adapter_dir, run_dir, rows, split_name="val", batch_size=16, full_metrics=False):
    metrics_path = run_dir / (f"{split_name}_metrics_full.json" if full_metrics else f"{split_name}_metrics.json")
    if metrics_path.exists():
        return json.load(open(metrics_path))

    pred_dir = run_dir / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    pred_csv = pred_dir / f"{split_name}_predictions.csv"

    predictions = pd.read_csv(pred_csv).to_dict("records") if pred_csv.exists() else []
    done_ids = {str(r["image_id"]) for r in predictions}
    remaining = [r for r in rows if str(r["image_id"]) not in done_ids]

    if remaining:
        base_model, processor, _ = init_model_and_processor()
        inf_model = PeftModel.from_pretrained(base_model, str(adapter_dir)).to("cuda").eval()
        inf_model.config.use_cache = True

        for i in tqdm(range(0, len(remaining), batch_size), desc=f"Generating ({split_name})"):
            batch = remaining[i : i + batch_size]
            msgs = [
                [{"role": "user", "content": [
                    {"type": "image", "image": Image.open(r["image_path"]).convert("RGB")},
                    {"type": "text", "text": PROMPT},
                ]}]
                for r in batch
            ]
            inputs = processor.apply_chat_template(
                msgs, tokenize=True, add_generation_prompt=True, padding=True, return_dict=True, return_tensors="pt"
            ).to("cuda")

            with torch.inference_mode():
                out_ids = inf_model.generate(**inputs, max_new_tokens=128, do_sample=False)

            trimmed = [o[len(inp):] for inp, o in zip(inputs["input_ids"], out_ids)]
            captions = [
                t.strip()
                for t in processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            ]
            for r, cap in zip(batch, captions):
                predictions.append({
                    "image_id": str(r["image_id"]),
                    "prediction": cap,
                    "references": json.dumps(r["references"]),
                })
            pd.DataFrame(predictions).to_csv(pred_csv, index=False)

        del inf_model, base_model
        gc.collect()
        torch.cuda.empty_cache()

    df = pd.read_csv(pred_csv)
    preds = df["prediction"].astype(str).tolist()
    refs = [json.loads(x) for x in df["references"]]

    tokenizer = PTBTokenizer()
    gts = tokenizer.tokenize({i: [{"caption": c} for c in r] for i, r in enumerate(refs)})
    res = tokenizer.tokenize({i: [{"caption": p}] for i, p in enumerate(preds)})

    cider_s, _ = Cider().compute_score(gts, res)
    results = {
        "CIDEr": float(cider_s),
        "avg_pred_words": float(np.mean([len(p.split()) for p in preds])),
    }

    if full_metrics:
        bleu_s, _ = Bleu(4).compute_score(gts, res)
        for idx, b in enumerate(bleu_s, 1):
            results[f"Bleu_{idx}"] = float(b)
        rouge_s, _ = Rouge().compute_score(gts, res)
        results["ROUGE_L"] = float(rouge_s)
        results["METEOR"] = float(compute_meteor_clean(gts, res))

        bert_P, bert_R, bert_F1 = bertscore_score(
            cands=preds, refs=refs, lang="en", model_type="roberta-large",
            device="cuda", batch_size=16, verbose=False, rescale_with_baseline=False,
        )
        results["BERTScore_P"] = float(bert_P.mean().item())
        results["BERTScore_R"] = float(bert_R.mean().item())
        results["BERTScore_F1"] = float(bert_F1.mean().item())

    with open(metrics_path, "w") as f:
        json.dump(results, f, indent=2)
    return results

In [ ]:
# 1. Train & Evaluate All Candidates on Validation Split
val_summary = []

for cfg in SWEEP_CONFIGS:
    lr, beta, ep = cfg["learning_rate"], cfg["beta"], cfg["num_epochs"]
    run_tag = f"{EXP_NAME}_lr{lr}_beta{beta}_ep{ep}"
    run_dir = EXP_ROOT / run_tag
    adapter_dir = run_dir / "adapter"
    run_dir.mkdir(parents=True, exist_ok=True)

    if not (adapter_dir / "adapter_config.json").exists():
        effective_bs = MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
        est_steps = math.ceil(len(train_dataset) / effective_bs) * ep

        model, processor, lora_config = init_model_and_processor()
        vision_collator = DataCollatorForVisionPreference(processor=processor, max_length=None)

        training_args = DPOConfig(
            output_dir=str(run_dir / "checkpoints"),
            num_train_epochs=ep,
            per_device_train_batch_size=MICRO_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=lr,
            beta=beta,
            loss_type="sigmoid",
            lr_scheduler_type="cosine",
            warmup_steps=int(0.05 * est_steps),
            optim="adamw_torch",
            max_grad_norm=0.3,
            bf16=True,
            tf32=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            max_length=None,
            logging_steps=10,
            save_strategy="no",
            remove_unused_columns=False,
            dataloader_num_workers=2,
            dataloader_pin_memory=True,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        )

        trainer = DPOTrainer(
            model=model,
            ref_model=None,
            args=training_args,
            train_dataset=train_dataset,
            processing_class=processor,
            peft_config=lora_config,
            data_collator=vision_collator,
        )

        torch.cuda.empty_cache()
        train_result = trainer.train()
        trainer.save_model(str(adapter_dir))
        processor.save_pretrained(str(adapter_dir))

        with open(run_dir / "experiment_config.json", "w") as f:
            json.dump({**cfg, "run_tag": run_tag, "seed": SEED}, f, indent=2)
        with open(run_dir / "training_metrics.json", "w") as f:
            json.dump(train_result.metrics, f, indent=2, default=str)

        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()

    val_res = eval_adapter_split(adapter_dir, run_dir, val_rows, split_name="val", full_metrics=True)
    val_summary.append({
        "Run": run_tag,
        "CIDEr": round(val_res["CIDEr"] * 100, 4),
        "Bleu_1": round(val_res["Bleu_1"] * 100, 4),
        "Bleu_2": round(val_res["Bleu_2"] * 100, 4),
        "Bleu_3": round(val_res["Bleu_3"] * 100, 4),
        "Bleu_4": round(val_res["Bleu_4"] * 100, 4),
        "METEOR": round(val_res["METEOR"] * 100, 4),
        "ROUGE_L": round(val_res["ROUGE_L"] * 100, 4),
        "BERTScore_F1": round(val_res["BERTScore_F1"] * 100, 4),
        "Words": round(val_res["avg_pred_words"], 1),
        "adapter_dir": str(adapter_dir),
        "run_dir": str(run_dir),
    })

val_df = pd.DataFrame(val_summary).sort_values("CIDEr", ascending=False).reset_index(drop=True)
val_df.drop(columns=["adapter_dir", "run_dir"]).to_csv(EXP_ROOT / "all_e3_val_comparison.csv", index=False)
display(val_df.drop(columns=["adapter_dir", "run_dir"]))

# 2. Evaluate Winning Model on Official KALE Test Set
winner = val_df.iloc[0]
print(f"\nWinner selected by Validation CIDEr: {winner['Run']} ({winner['CIDEr']:.2f})")

test_res = eval_adapter_split(
    Path(winner["adapter_dir"]),
    Path(winner["run_dir"]),
    test_rows,
    split_name="test",
    full_metrics=True,
)

pct_metrics = {"Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr", "BERTScore_P", "BERTScore_R", "BERTScore_F1"}
final_df = pd.DataFrame([
    {"Metric": k, "Raw Score": v, "Reported Score": v * 100 if k in pct_metrics else v}
    for k, v in test_res.items()
])
final_df.to_csv(Path(winner["run_dir"]) / "final_winner_test_metrics.csv", index=False)
display(final_df.style.format({"Raw Score": "{:.4f}", "Reported Score": "{:.4f}"}))